<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [17]</a>'.</span>

In [1]:
"""
fast_compare_osm_correct.py
────────────────────────────
Fast + correct evaluation of predicted GeoJSONs vs OSM ground truth.

Fixes:
- Correct precision/recall definition (bounded [0,1])
- Symmetric matching (GT ↔ Pred)
- STRtree acceleration
- No unary_union (RAM-safe)
"""

'\nfast_compare_osm_correct.py\n────────────────────────────\nFast + correct evaluation of predicted GeoJSONs vs OSM ground truth.\n\nFixes:\n- Correct precision/recall definition (bounded [0,1])\n- Symmetric matching (GT ↔ Pred)\n- STRtree acceleration\n- No unary_union (RAM-safe)\n'

In [2]:
import os
import warnings

In [3]:
warnings.filterwarnings("ignore")

In [4]:
import numpy as np
import geopandas as gpd
import osmnx as ox
from shapely.strtree import STRtree
from tqdm import tqdm

In [5]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
PLACE = "El Harrach, Algeria"
TARGET_CRS = "EPSG:3857"
LINE_BUFFER_M = 3

In [6]:
PREDICTED = {
    "building": "output/vect/poly/building.geojson",
    "water": "output/vect/poly/water.geojson",
    "residential area": "output/vect/poly/residential area.geojson",
    "grass": "output/vect/poly/grass.geojson",
    "autoroute": "output/vect/line/autoroute.geojson",
    "route_nationale": "output/vect/line/route_nationale.geojson",
    "street": "output/vect/line/street.geojson",
    "railway": "output/vect/line/railway.geojson",
}

In [7]:
LINE_CLASSES = {"autoroute", "route_nationale", "street", "railway"}

In [8]:
# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def load_pred(path, bbox=None):
    if not os.path.exists(path):
        return None

    gdf = gpd.read_file(path)
    if gdf.empty:
        return None

    if gdf.crs is None:
        gdf = gdf.set_crs(TARGET_CRS)
    else:
        gdf = gdf.to_crs(TARGET_CRS)

    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty]

    if bbox is not None:
        try:
            gdf = gpd.clip(gdf, bbox)
        except:
            pass

    return gdf

In [9]:
def build_tree(geoms):
    geoms = list(geoms)
    if not geoms:
        return None, []
    return STRtree(geoms), geoms

In [10]:
# ─────────────────────────────────────────────
# CORRECT METRICS (SYMMETRIC)
# ─────────────────────────────────────────────
def symmetric_match_metrics(pred_gdf, gt_gdf):
    """
    Correct evaluation:
    - precision = correct_pred / total_pred
    - recall    = correct_gt / total_gt
    - F1        = harmonic mean

    Matching rule:
    - A feature is "correct" if it intersects ANY counterpart
    """

    if pred_gdf is None or gt_gdf is None:
        return 0.0, 0.0, 0.0

    if pred_gdf.empty or gt_gdf.empty:
        return 0.0, 0.0, 0.0

    pred_geoms = list(pred_gdf.geometry)
    gt_geoms = list(gt_gdf.geometry)

    pred_tree, pred_list = build_tree(pred_geoms)
    gt_tree, gt_list = build_tree(gt_geoms)

    if pred_tree is None or gt_tree is None:
        return 0.0, 0.0, 0.0

    # ────────────────
    # GT → Pred (recall)
    # ────────────────
    gt_hits = 0
    for g in gt_list:
        if g.is_empty:
            continue
        idxs = pred_tree.query(g)
        if any(g.intersects(pred_list[i]) for i in idxs):
            gt_hits += 1

    # ────────────────
    # Pred → GT (precision)
    # ────────────────
    pred_hits = 0
    for p in pred_list:
        if p.is_empty:
            continue
        idxs = gt_tree.query(p)
        if any(p.intersects(gt_list[i]) for i in idxs):
            pred_hits += 1

    recall = gt_hits / len(gt_list) if gt_list else 0.0
    precision = pred_hits / len(pred_list) if pred_list else 0.0

    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return precision, recall, f1

In [11]:
# ─────────────────────────────────────────────
# STEP 1 — BOUNDARY
# ─────────────────────────────────────────────
print("=" * 60)
print(" FAST OSM EVALUATION (CORRECT METRICS)")
print("=" * 60)

 FAST OSM EVALUATION (CORRECT METRICS)


In [12]:
boundary = ox.geocode_to_gdf(PLACE).to_crs(TARGET_CRS)
bbox = boundary.geometry.iloc[0]

In [13]:
# ─────────────────────────────────────────────
# STEP 2 — OSM DATA
# ─────────────────────────────────────────────
print("\n[1/3] Loading OSM ground truth...")


[1/3] Loading OSM ground truth...


In [14]:
gt = {}

In [15]:
def fetch(tags, name):
    try:
        gdf = ox.features_from_place(PLACE, tags)
        gdf = gdf.to_crs(TARGET_CRS)
        gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty]
        gdf = gpd.clip(gdf, bbox)
        gt[name] = gdf
        print(f"  {name:<18}: {len(gdf)}")
    except Exception as e:
        print(f"  {name:<18}: FAILED ({e})")

In [16]:
fetch({"building": True}, "building")
fetch({"natural": ["water", "wetland"], "waterway": True}, "water")
fetch({"landuse": ["residential"]}, "residential area")
fetch({"landuse": ["grass", "meadow", "park"], "natural": "grassland"}, "grass")

  building          : 4782
  water             : 10
  residential area  : 9


  grass             : 9


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [17]:
# roads
try:
    G = ox.graph_from_place(PLACE, network_type="drive")
    edges = ox.graph_to_gdfs(G, nodes=False).to_crs(TARGET_CRS)

    highway_map = {
        "autoroute": ["motorway", "trunk"],
        "route_nationale": ["primary", "secondary"],
        "street": ["tertiary", "residential", "service"],
    }

    for k, v in highway_map.items():
        subset = edges[edges["highway"].astype(str).isin(v)]
        subset = gpd.clip(subset, bbox)
        gt[k] = subset
        print(f"  {k:<18}: {len(subset)}")

_IncompleteInputError: incomplete input (2092891840.py, line 16)

In [ ]:
except Exception as e:
    print("roads failed:", e)

In [ ]:
fetch({"railway": ["rail", "tram", "subway"]}, "railway")

In [ ]:
# ─────────────────────────────────────────────
# STEP 3 — EVALUATION
# ─────────────────────────────────────────────
print("\n[2/3] Computing metrics...\n")

In [ ]:
results = []

In [ ]:
for cls, path in tqdm(PREDICTED.items(), desc="Classes"):

    pred = load_pred(path, bbox)
    is_line = cls in LINE_CLASSES

    if pred is None:
        results.append((cls, "MISSING PRED", 0, 0, 0))
        continue

    if cls not in gt:
        results.append((cls, "MISSING GT", 0, 0, 0))
        continue

    gt_gdf = gt[cls]

    # buffer lines for fairness
    if is_line:
        pred = pred.copy()
        gt_gdf = gt_gdf.copy()
        pred["geometry"] = pred.buffer(LINE_BUFFER_M)
        gt_gdf["geometry"] = gt_gdf.buffer(LINE_BUFFER_M)

    precision, recall, f1 = symmetric_match_metrics(pred, gt_gdf)

    results.append((cls, "OK", precision, recall, f1))

In [ ]:
# ─────────────────────────────────────────────
# STEP 4 — REPORT
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"{'CLASS':<20} {'PREC':>8} {'REC':>8} {'F1':>8}")
print("-" * 60)

In [ ]:
f1_scores = []

In [ ]:
for cls, status, p, r, f in results:
    if status != "OK":
        print(f"{cls:<20} {status}")
        continue

    f1_scores.append(f)

    print(f"{cls:<20} {p*100:7.1f}% {r*100:7.1f}% {f*100:7.1f}%")

In [ ]:
print("-" * 60)

In [ ]:
if f1_scores:
    print(f"{'AVERAGE':<20} {'':8} {'':8} {np.mean(f1_scores)*100:7.1f}%")

In [ ]:
print("=" * 60)
print("Done.")